# Recurrent Neural Networks
You should build an end-to-end machine learning pipeline using a recurrent neural network model. In particular, you should do the following:
- Load the `jena climate` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the [keras repository](https://keras.io/examples/timeseries/timeseries_weather_forecasting/).
- Split the dataset into training, validation, and test sets. Note that you cannot split time series using [Scikit-Learn](https://keras.io/examples/timeseries/timeseries_weather_forecasting/).
- Build an end-to-end machine learning pipeline, including a [recurrent neural network](https://keras.io/examples/timeseries/timeseries_weather_forecasting/) model.
- Optimize your pipeline by validating your design decisions.
- Test the best pipeline on the test set and report various [evaluation metrics](https://scikit-learn.org/0.15/modules/model_evaluation.html).  
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

In [1]:
!wget https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip
!unzip jena_climate_2009_2016.csv.zip

--2025-06-03 12:12:32--  https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 172.217.204.207, 172.217.203.207, 142.250.98.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.217.204.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13568290 (13M) [application/zip]
Saving to: ‘jena_climate_2009_2016.csv.zip.11’

jena_climate_2009_2 100%[===================>]  12.94M  --.-KB/s    in 0.06s   

2025-06-03 12:12:32 (199 MB/s) - ‘jena_climate_2009_2016.csv.zip.11’ saved [13568290/13568290]

Archive:  jena_climate_2009_2016.csv.zip
replace jena_climate_2009_2016.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: jena_climate_2009_2016.csv  


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [3]:
df = pd.read_csv('jena_climate_2009_2016.csv')
df.head()

,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


In [4]:
df.shape

(420551, 15)

In [5]:
df['Date Time'] = pd.to_datetime(df['Date Time'], format='%d.%m.%Y %H:%M:%S')
df_midnight = df[(df['Date Time'].dt.hour == 0) & (df['Date Time'].dt.minute == 0)]

In [6]:
df_train = df_midnight[:int(0.5 * len(df_midnight))]
df_val = df_midnight[int(0.5 * len(df_midnight)):int(0.75 * len(df_midnight))]
df_test = df_midnight[int(0.75 * len(df_midnight)):]

print(f"Dataset size: {df_midnight.shape}")
print(f"Training dataset size: {df_train.shape}")
print(f"Validation dataset size: {df_val.shape}")
print(f"Test dataset size: {df_test.shape}")

Dataset size: (2920, 15)
Training dataset size: (1460, 15)
Validation dataset size: (730, 15)
Test dataset size: (730, 15)


In [7]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1460 entries, 143 to 210237
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date Time        1460 non-null   datetime64[ns]
 1   p (mbar)         1460 non-null   float64       
 2   T (degC)         1460 non-null   float64       
 3   Tpot (K)         1460 non-null   float64       
 4   Tdew (degC)      1460 non-null   float64       
 5   rh (%)           1460 non-null   float64       
 6   VPmax (mbar)     1460 non-null   float64       
 7   VPact (mbar)     1460 non-null   float64       
 8   VPdef (mbar)     1460 non-null   float64       
 9   sh (g/kg)        1460 non-null   float64       
 10  H2OC (mmol/mol)  1460 non-null   float64       
 11  rho (g/m**3)     1460 non-null   float64       
 12  wv (m/s)         1460 non-null   float64       
 13  max. wv (m/s)    1460 non-null   float64       
 14  wd (deg)         1460 non-null   float64 

In [8]:
df_train.describe()

,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
count,1460,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,2010-12-31 20:59:30.410958848,988.994110,7.243548,281.297842,4.209849,81.990760,11.387123,9.122384,2.264705,5.763425,9.226897,1225.392979,1.699493,2.751877,185.089500
min,2009-01-02 00:00:00,952.330000,-21.090000,252.470000,-23.010000,42.750000,1.130000,0.950000,0.000000,0.590000,0.950000,1142.580000,0.000000,0.000000,0.000000
25%,2010-01-01 18:00:00,983.967500,2.045000,276.227500,-0.455000,74.600000,7.077500,5.907500,0.677500,3.750000,6.010000,1197.717500,0.770000,1.400000,163.750000
50%,2010-12-31 12:00:00,989.630000,8.085000,282.190000,4.730000,83.700000,10.800000,8.565000,1.570000,5.430000,8.700000,1220.090000,1.340000,2.250000,204.600000
75%,2011-12-31 06:00:00,994.422500,12.930000,287.100000,9.547500,91.125000,14.930000,11.925000,3.172500,7.505000,12.015000,1248.160000,2.262500,3.520000,234.525000
max,2012-12-30 00:00:00,1011.910000,24.310000,298.150000,18.760000,100.000000,30.450000,21.690000,13.040000,13.770000,21.960000,1381.560000,10.970000,16.140000,359.500000
std,NaN,8.452016,7.637388,7.745210,7.064406,11.294365,5.404897,4.091730,2.149438,2.596520,4.142257,38.164431,1.351278,1.965531,79.473744


In [9]:
x_train = df_train.drop(["T (degC)"], axis=1)
y_train = df_train["T (degC)"]

x_val = df_val.drop(["T (degC)"], axis=1)
y_val = df_val["T (degC)"]

x_test = df_test.drop(["T (degC)"], axis=1)
y_test = df_test["T (degC)"]

print(f"Shape of x_train:- {x_train.shape}")
print(f"Shape of x_test:- {x_test.shape}")
print(f"Shape of x_val:- {x_val.shape}")
print(f"Shape of y_val:- {y_val.shape}")
print(f"Shape of y_train:- {y_train.shape}")
print(f"Shape of y_test:- {y_test.shape}")

Shape of x_train:- (1460, 14)
Shape of x_test:- (730, 14)
Shape of x_val:- (730, 14)
Shape of y_val:- (730,)
Shape of y_train:- (1460,)
Shape of y_test:- (730,)


In [10]:
numerical_attributes = x_train.select_dtypes(include=['float64']).columns

ct = ColumnTransformer([("scaling", MinMaxScaler(), numerical_attributes)])
ct.fit(x_train)

x_train = ct.transform(x_train)
x_val = ct.transform(x_val)
x_test = ct.transform(x_test)

In [11]:
dataset_train = keras.preprocessing.timeseries_dataset_from_array(x_train, y_train, sequence_length=7, sampling_rate=6, batch_size=256)
dataset_val = keras.preprocessing.timeseries_dataset_from_array(x_val, y_val, sequence_length=7, sampling_rate=6, batch_size=256)

In [12]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_mae', patience=10, restore_best_weights=True)

In [13]:
model = keras.Sequential([
    layers.Input(shape=(100, x_train.shape[1],)),
    layers.LSTM(128, return_sequences=True),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(32),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100, 128)       │        72,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100, 64)        │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 149,121 (582.50 KB)

 Trainable params: 149,121 (582.50 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
path_checkpoint = "model_checkpoint.weights.h5"
es_callback = keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0, patience=10)

modelckpt_callback = keras.callbacks.ModelCheckpoint(
    monitor="val_loss",
    filepath=path_checkpoint,
    verbose=1,
    save_weights_only=True,
    save_best_only=True,
)

history = model.fit(
    dataset_train,
    epochs=100,
    validation_data=dataset_val,
    callbacks=[es_callback, modelckpt_callback],
)

Epoch 1/100
5/6 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - loss: 106.7650 - mae: 8.7548 
Epoch 1: val_loss improved from inf to 118.89619, saving model to model_checkpoint.weights.h5
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 314ms/step - loss: 108.1492 - mae: 8.8155 - val_loss: 118.8962 - val_mae: 9.3082
Epoch 2/100
5/6 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 102.2689 - mae: 8.5617
Epoch 2: val_loss improved from 118.89619 to 105.53217, saving model to model_checkpoint.weights.h5
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 156ms/step - loss: 103.0807 - mae: 8.5949 - val_loss: 105.5322 - val_mae: 8.7258
Epoch 3/100
5/6 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 90.4992 - mae: 8.0490 
Epoch 3: val_loss improved from 105.53217 to 81.86179, saving model to model_checkpoint.weights.h5
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step - loss: 90.3832 - mae: 8.0305 - val_loss: 81.8618 - val_mae: 7.6242
Epoch 4/100
5/6 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - loss: 72.9775 - mae: 7.2254
Epoch 4: val_loss improved from 81.86179 to 56.75058, sav